# RAG Q&A with Citations

**Retrieval-Augmented Generation, built entirely with open-source tools.**

You will build a question-answering system that reads *your* documents and returns
answers **grounded in the sources**, with inline citations like `[1]`, `[2]`. By the
end you will have a reusable `ask()` pipeline that does hybrid retrieval, reranking,
grounded generation, and a faithfulness check.

## What you learn
- Chunking a corpus and tracking metadata for citations
- Embeddings + vector search (dense retrieval)
- Sparse retrieval (BM25) and **hybrid fusion** (Reciprocal Rank Fusion)
- **Reranking** with a cross-encoder
- Grounded prompting that forces citations and "I don't know"
- **Faithfulness evaluation** with an LLM-as-judge

## Open-source stack (no paid APIs)
| Concern | Tool | License |
|---|---|---|
| Local LLM | **transformers** (`Qwen/Qwen2.5-1.5B-Instruct`, runs on CPU) | Apache-2.0 |
| Embeddings | **sentence-transformers** (`BAAI/bge-small-en-v1.5`) | Apache-2.0 |
| Vector store | **Chroma** | Apache-2.0 |
| Sparse search | **rank-bm25** | Apache-2.0 |
| Reranker | **sentence-transformers CrossEncoder** | Apache-2.0 |
| PDF parsing | **pypdf** | BSD |

## Prerequisites
- Python 3.10–3.12 (any version with `torch` wheels available)
- `pip install -r requirements.txt` (Step 0). Everything runs **locally on CPU** —
  no API keys and no separate model server. The first run downloads a few hundred MB
  of open model weights from the Hugging Face Hub.
- *Optional:* point `generate()` at a local [Ollama](https://ollama.com) server for
  larger / faster models — see the commented note in the LLM setup cell.


## Step 0 — Environment setup

Install the dependencies, then load a small open-source, instruction-tuned LLM that
runs locally on CPU. Run this once; restart the kernel if a fresh install of `torch`
or `chromadb` was pulled in. The first model load downloads the weights.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# Load a small, open-source, instruction-tuned LLM that runs locally on CPU.
from transformers import pipeline

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Apache-2.0, runs on CPU. Smaller/faster: "Qwen/Qwen2.5-0.5B-Instruct"

llm = pipeline("text-generation", model=MODEL)
# We decode greedily; unset the model's sampling defaults to keep the output clean.
for _p in ("temperature", "top_p", "top_k"):
    setattr(llm.model.generation_config, _p, None)


def generate(messages, max_new_tokens: int = 200) -> str:
    "Run a chat completion locally and return the assistant's reply text."
    out = llm(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]


print("Loaded", MODEL)

# --- Optional: use a local Ollama server instead (larger models, GPU-friendly) ---
# import ollama
# def generate(messages, max_new_tokens=200):
#     r = ollama.chat(model="llama3.1:8b", messages=messages,
#                     options={"temperature": 0.0, "num_predict": max_new_tokens})
#     return r["message"]["content"]

## Step 1 — Build a sample corpus

So the notebook is self-contained, we write a small set of documents about a
fictional product, **Nimbus Analytics**. Later, drop your own `.md`, `.txt`, or
`.pdf` files into the `corpus/` folder and re-run from Step 2.

In [ ]:
from pathlib import Path

CORPUS = Path("corpus")
CORPUS.mkdir(exist_ok=True)

docs = {
    "product_overview.md": """
Nimbus Analytics is a self-hosted product analytics platform.
The Free tier allows up to 3 team members and 1 million tracked events per month.
The Pro and Enterprise tiers remove these limits.
Nimbus is released under the Apache 2.0 open-source license.
""",
    "billing_faq.md": """
Nimbus billing runs monthly. The Free tier costs nothing and never expires.
Pro costs 49 USD per month per project. Enterprise pricing is custom.
Invoices are issued on the first day of each month.
Customers who pay annually receive a 20 percent discount.
""",
    "security_policy.md": """
Nimbus supports single sign-on (SSO) via SAML and OIDC on the Enterprise tier.
Event data is retained for 90 days on the Free tier and 24 months on paid tiers.
Customer data can be stored in the EU (Frankfurt) or the US (Virginia) region.
All data is encrypted at rest with AES-256.
""",
}

for name, text in docs.items():
    (CORPUS / name).write_text(text.strip() + "\n", encoding="utf-8")

print("Wrote", len(docs), "documents to", CORPUS.resolve())

## Step 2 — Load and chunk documents

Split each document into overlapping word-windows. We keep the **source filename**
and a **chunk id** on every chunk — that metadata is what makes citations possible.

In [ ]:
from dataclasses import dataclass
from pypdf import PdfReader


@dataclass
class Chunk:
    text: str
    source: str
    chunk_id: int


def read_file(path: Path) -> str:
    if path.suffix.lower() == ".pdf":
        return "\n".join((page.extract_text() or "") for page in PdfReader(str(path)).pages)
    return path.read_text(encoding="utf-8")


def chunk_text(text: str, size: int = 120, overlap: int = 30):
    words = text.split()
    step = max(1, size - overlap)
    return [" ".join(words[i:i + size]) for i in range(0, len(words), step)] or [text]


chunks, cid = [], 0
for path in sorted(CORPUS.glob("*")):
    if path.suffix.lower() not in {".md", ".txt", ".pdf"}:
        continue
    for piece in chunk_text(read_file(path)):
        chunks.append(Chunk(text=piece, source=path.name, chunk_id=cid))
        cid += 1

print(f"Created {len(chunks)} chunks from {len(list(CORPUS.glob('*')))} files")
print("Example chunk ->", chunks[0].source, "::", chunks[0].text[:80], "...")

## Step 3 — Embed the chunks

`bge-small-en-v1.5` is a small, fast, open embedding model. We normalize the
vectors so cosine similarity is just a dot product. The first run downloads the
model weights (~130 MB).

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

texts = [c.text for c in chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
print("Embedding matrix shape:", embeddings.shape)

## Step 4 — Build a persistent vector store (Chroma)

We index the chunk text, embeddings, and metadata. `PersistentClient` writes to
`./chroma_db`, so the index survives a kernel restart.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

# Start clean so re-running the notebook does not duplicate rows.
try:
    client.delete_collection("docs")
except Exception:
    pass

collection = client.create_collection("docs", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[str(c.chunk_id) for c in chunks],
    documents=texts,
    embeddings=[e.tolist() for e in embeddings],
    metadatas=[{"source": c.source, "chunk_id": c.chunk_id} for c in chunks],
)
print("Indexed", collection.count(), "chunks")

## Step 5 — Dense retrieval

Embed the query, ask Chroma for the nearest chunks, and convert cosine distance
into a similarity score.

In [ ]:
def dense_retrieve(query: str, k: int = 4):
    q = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = collection.query(query_embeddings=[q], n_results=k)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": doc, "source": meta["source"], "score": 1 - dist})
    return hits


for h in dense_retrieve("What are the free tier limits?"):
    print(f"[{h['source']}] sim={h['score']:.3f}  {h['text'][:70]}...")

## Step 6 — A grounded, citation-ready prompt

The system prompt is where grounding is enforced: answer **only** from the numbered
sources, cite every claim with `[n]`, and refuse when the sources do not cover the
question. This is the single biggest lever against hallucination.

In [ ]:
def build_context(hits) -> str:
    return "\n\n".join(
        f"[{i}] (source: {h['source']})\n{h['text']}" for i, h in enumerate(hits, 1)
    )


SYSTEM = (
    "You are a precise assistant. Answer ONLY using the numbered sources below. "
    "Cite every claim with its source number in square brackets, e.g. [1]. "
    "If the sources do not contain the answer, reply exactly: "
    "'I don't know based on the provided documents.'"
)


def build_messages(question: str, hits):
    user = f"Sources:\n{build_context(hits)}\n\nQuestion: {question}\n\nAnswer with citations:"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

## Step 7 — Generate a grounded answer (local LLM)

Greedy decoding (`do_sample=False`, set in the loader above) keeps the answer
faithful to the retrieved text.

In [ ]:
q = "What are the Nimbus free tier limits, and how long is data retained?"
hits = dense_retrieve(q, k=4)
answer = generate(build_messages(q, hits))

print(answer)
print("\nSources")
for i, h in enumerate(hits, 1):
    print(f"  [{i}] {h['source']}")

## Step 8 — Hybrid retrieval (BM25 + dense)

Dense search captures meaning; BM25 captures exact keywords (product names, IDs,
error codes). **Reciprocal Rank Fusion** blends both ranked lists without needing
to normalize their scores.

In [ ]:
from rank_bm25 import BM25Okapi

tokenized = [c.text.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized)


def bm25_retrieve(query: str, k: int = 8):
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [{"text": chunks[i].text, "source": chunks[i].source, "score": float(scores[i])} for i in ranked]


def rrf(*ranked_lists, k: int = 60):
    fused = {}
    for lst in ranked_lists:
        for rank, h in enumerate(lst):
            entry = fused.setdefault(h["text"], {"hit": h, "score": 0.0})
            entry["score"] += 1.0 / (k + rank + 1)
    ordered = sorted(fused.values(), key=lambda x: x["score"], reverse=True)
    return [{**e["hit"], "score": e["score"]} for e in ordered]


def hybrid_retrieve(query: str, k: int = 6):
    return rrf(dense_retrieve(query, k=8), bm25_retrieve(query, k=8))[:k]


for h in hybrid_retrieve("Does Nimbus support SAML SSO?"):
    print(f"[{h['source']}] rrf={h['score']:.4f}  {h['text'][:60]}...")

## Step 9 — Rerank with a cross-encoder

A cross-encoder reads the query and each candidate **together**, giving a far more
accurate relevance score than embedding similarity. We over-retrieve with hybrid
search, then keep only the top few after reranking.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, hits, top_n: int = 4):
    scores = reranker.predict([(query, h["text"]) for h in hits])
    for h, s in zip(hits, scores):
        h["rerank_score"] = float(s)
    return sorted(hits, key=lambda h: h["rerank_score"], reverse=True)[:top_n]


ranked = rerank("How long is event data kept?", hybrid_retrieve("How long is event data kept?"))
for h in ranked:
    print(f"[{h['source']}] rerank={h['rerank_score']:.3f}  {h['text'][:60]}...")

## Step 10 — The full `ask()` pipeline

Hybrid retrieve -> rerank -> grounded generate -> return the answer plus the exact
sources used. This is the reusable function you would put behind an API endpoint.

In [ ]:
def ask(question: str, k_retrieve: int = 8, k_final: int = 4, show_sources: bool = True):
    candidates = hybrid_retrieve(question, k=k_retrieve)
    hits = rerank(question, candidates, top_n=k_final)
    answer = generate(build_messages(question, hits))
    if show_sources:
        answer += "\n\nRetrieved sources:\n" + "\n".join(
            f"  [{i}] {h['source']} (rerank={h.get('rerank_score', 0):.2f})"
            for i, h in enumerate(hits, 1)
        )
    return answer, hits


ans, hits = ask("Does Nimbus support SSO, and which regions can store my data?")
print(ans)

## Step 11 — Evaluate faithfulness (LLM-as-judge)

A RAG answer is only useful if it is **supported by the sources**. We ask the same
local model to act as a strict fact-checker and return a JSON verdict. The cell prints
the parsed `supported` flag, itemizes any **unsupported claims** verbatim, and echoes
the raw judge output — so a `False` always tells you *which* sentence failed.

In [ ]:
import json, re

JUDGE = (
    "You are a strict fact-checker. Compare the ANSWER against the SOURCES. "
    "A claim counts as UNSUPPORTED if the sources do not explicitly state it, even when it "
    "sounds plausible or reuses similar wording. Respond ONLY with JSON of the form: "
    '{"supported": true|false, "unsupported_claims": ["<verbatim phrase from the answer>"]}. '
    "List every unsupported claim verbatim; use an empty list only if the whole answer is supported."
)


def faithfulness(answer: str, hits):
    messages = [
        {"role": "system", "content": JUDGE},
        {"role": "user", "content": f"SOURCES:\n{build_context(hits)}\n\nANSWER:\n{answer}\n\nJSON verdict:"},
    ]
    raw = generate(messages)
    verdict = {"supported": None, "unsupported_claims": [], "raw_judge": raw.strip()}
    match = re.search(r"\{.*\}", raw, re.S)
    if match:
        try:
            parsed = json.loads(match.group(0))
            verdict["supported"] = parsed.get("supported")
            verdict["unsupported_claims"] = parsed.get("unsupported_claims") or []
        except json.JSONDecodeError:
            pass
    return verdict


ans, hits = ask("What is the Nimbus data retention period?", show_sources=False)
print("Answer:", ans, "\n")

verdict = faithfulness(ans, hits)
print("supported:", verdict["supported"])
if verdict["unsupported_claims"]:
    print("unsupported claims:")
    for claim in verdict["unsupported_claims"]:
        print("  -", claim)
elif verdict["supported"] is False:
    print("(judge flagged the answer but did not itemize a claim — see raw output)")
print("\nraw judge output:")
print(verdict["raw_judge"])

## Step 12 — Stretch goals & what you learned

You built a complete open-source RAG system: chunking with citation metadata,
dense + sparse hybrid retrieval, cross-encoder reranking, grounded generation, and
an automated faithfulness check.

**Stretch goals**
- Swap the sample corpus for real PDFs and tune `size` / `overlap`.
- Add **query rewriting**: ask the LLM to expand the question before retrieval.
- Add a "no answer" gate: if the top rerank score is below a threshold, refuse.
- Measure retrieval quality with a small labeled set (recall@k, MRR).
- Wrap `ask()` behind a small web API so other services can query your corpus.
